In [2]:
import itertools
import concurrent.futures
import constants as Cs
import os
import json
import datetime
import pickle
import optuna

SEEDS = list(range(101, 102))
TEST_EVAL_EPS = 5
# lambda fit archving [FrozenTrial(number=80, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 4, 53, 822616), datetime_complete=datetime.datetime(2026, 5, 22, 5, 36, 8, 359073), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=81, value=None), FrozenTrial(number=86, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 36, 31, 908497), datetime_complete=datetime.datetime(2026, 5, 22, 6, 11, 5, 128793), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=87, value=None)]
# lambda - this one is bad actually novelty [FrozenTrial(number=4, state=<TrialState.COMPLETE: 1>, values=[-9.291787437438707], datetime_start=datetime.datetime(2026, 5, 19, 22, 25, 13, 886515), datetime_complete=datetime.datetime(2026, 5, 19, 23, 0, 16, 677521), params={'crossmethod': 'mean', 'lambda': 60, 'mu': 60, 'mutation_rate': 0.18, 'cross_rate': 1.0, 'sigma': 2.5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5)}, trial_id=208, value=None)]
EXAMPLE_MAPPING  = {
    "lambda":"l",
    "mu": "m",
    "mutation_rate":"mr",
    "cross_rate":"cr",
    "sigma": "mutation_sigma",
    "cross":"cross_uni",
    "crossmethod":"cross_method"
}

def rename(dict):
    return {EXAMPLE_MAPPING.get(k, k): v for k, v in dict.items()}

def task_job(en, alg, container, args, s, out_path):
    env = Cs.ENIVROMENTS[en]()

    df, pop = Cs.ALG_MAPPING[alg].argumented_function(env=en, container=container, seed=s, out_path=out_path, **args)
    print("Testing " + str(s))
    fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for p in pop]
    print("Finished seed %d of algorithm %s" % (s, alg))
    return fitnesses, pop


def evaluation_of_setup(en, alg, container, out_path, **kwargs):
    #we do not deviate the sigma
    # first we evaluate the current setup

    stat_futures = {}
    args = rename(kwargs)
    args["ng"] = 15
    dirpath = os.path.join(os.path.realpath(out_path), container,alg)

    with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
        print("Launching " + alg + "on Enviroment " + str(en))
        for s in SEEDS:
            future = executor.submit(task_job, container=container, alg=alg, en=en, args=args, s=s, out_path=dirpath + "/stat")
            stat_futures[future] = s

    stats = {}
    pops = {}
    for future in concurrent.futures.as_completed(stat_futures):
        s = stat_futures[future]
        result = future.result()
        if "fitness" not in stats:
            stats["fitness"] = {}
        stats["fitness"][s] = result[0]
        pops[s]= result[1] 
    ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"{ts}_{container}_"+ "|".join(f"{k}: {v}" for k, v in args.items())
    stats["environment"] = en
    stats["algorithm"] = alg
    stats["container"] = container
    stats["arguments"] = args
    json_path = os.path.join(dirpath, filename + ".json")
    pickle_path = os.path.join(dirpath, filename + ".plk")

    with open(json_path, "w") as json_file:
        json.dump(stats,json_file)
        print("Finished "+ filename)
    with open(pickle_path, "wb") as f:
        pickle.dump(pops, f)
    return pops, stats



2026-06-11 22:01:22.811452: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-11 22:01:22.841319: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-11 22:01:23.565637: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .auto

In [12]:
#new_study = optuna.load_study(study_name="fitarchiving",storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db")
storage = "sqlite:///Data/optuna/lunarlander/fitness/diff.db"
studies = optuna.get_all_study_summaries(
    storage=storage
)
print(studies)
for s in studies:
    print(s.study_name)
    print(s.datetime_start)
    print(s.n_trials)
    print(s.best_trial)
    print("--")

#new_study = optuna.load_study(study_name="no-name-50aac953-b263-4c39-8a27-62db0b093c9f",storage=storage)
#new_study.best_trials


[<optuna.study._study_summary.StudySummary object at 0x75d209562bc0>, <optuna.study._study_summary.StudySummary object at 0x75d209604130>]
no-name-6b4c1a63-b1ca-464e-82df-ae8138d205a2
2026-05-20 22:38:56.685430
5
None
--
no-name-1478d1bf-8dea-4995-829f-60e6bd3fc2c9
2026-05-20 22:40:11.178944
120
FrozenTrial(number=46, state=<TrialState.COMPLETE: 1>, values=[80.94444268266854], datetime_start=datetime.datetime(2026, 5, 21, 2, 6, 45, 482359), datetime_complete=datetime.datetime(2026, 5, 21, 2, 21, 19, 396043), params={'pop': 40, 'mutation_rate': 0.06, 'cross_rate': 0.3}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'pop': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1)}, trial_id=52, value=None)
--


In [ ]:
import numpy as np
def select_minimal_exaples(examples):
    pop = np.inf
    selected_trials = []
    selected_trials_index = []
    for i, t in enumerate(examples):
        if "lambda" in t:
            k = t["lambda"]
            if "mu" in t:
                k+=t["mu"]
        elif "pop" in t:
            k=t["pop"]
        else: raise NameError(f"wtf")
        if pop == k:
            selected_trials.append(t)
        elif pop > k:
            pop = k
            selected_trials = [t]
            selected_trials_index.append(i)
        return selected_trials


In [4]:
def make_final_examination(en, alg, container):
    with open("relevant_studies.json", "r") as f:
        relevant_study_names = json.load(f)
    storage = f"sqlite:///Data/optuna/{en}/{container}/{alg}.db"
    study_name = relevant_study_names[container][alg][en]

    new_study = optuna.load_study(study_name=study_name,storage=storage)
    most_promising = select_minimal_exaples([t.params for t in new_study.best_trials])
    pops, stats = evaluation_of_setup(
        en=en, 
        alg=alg, 
        container=container,
        out_path=f"./Data/final/{en}",
        **most_promising[0]
    )
    return pops, stats

pops, stats = make_final_examination(en="lunarlander", alg="lambda", container="fit_archiving")

Launching lambdaon Enviroment lunarlander


2026-06-01 14:27:02.006713: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-01 14:27:02.008531: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


gen	nevals	avg     	min     	max    	std    
0  	70    	-395.639	-841.759	29.5301	185.716
1  	62    	-210.042	-532.956	29.5301	137.673
2  	63    	-98.7229	-413.202	29.5301	88.3797
3  	62    	-58.5545	-279.117	29.5301	66.4457
4  	68    	-40.0555	-236.891	29.5301	63.8535
5  	64    	-2.4072 	-132.13 	59.5705	48.7242
6  	66    	19.4816 	-257.974	61.4435	44.9027
7  	65    	41.8377 	29.5301 	86.8277	16.5846
8  	63    	54.6917 	29.5301 	90.0312	19.9769
9  	69    	73.3907 	29.5301 	97.3222	18.9108
10 	64    	86.78   	32.1535 	101.98 	11.394 
11 	64    	92.2602 	47.5807 	102.393	8.34818
12 	65    	97.8256 	77.3512 	117.63 	6.55908
13 	61    	99.4937 	87.4237 	107.888	4.35499
14 	61    	101.819 	92.8736 	107.888	3.58416
15 	65    	102.625 	64.2507 	107.888	6.00928
Testing 101
Finished seed 101 of algorithm lambda


2026-06-01 14:54:09.757258: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-06-01 14:54:09.766245: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Finished 2026-06-01_14-54-10_fit_archiving_cross_method: uniform|l: 70|m: 70|mr: 0.01|cr: 0.9000000000000001|mutation_sigma: 2.5|archiving_period: 4|archive_batch: 2|cross_uni: 0.8|ng: 15


In [5]:
print(pops)

{101: [<deap.creator.Individual object at 0x775ceac65420>, <deap.creator.Individual object at 0x775ceac65660>, <deap.creator.Individual object at 0x775ceac0f370>, <deap.creator.Individual object at 0x775ceaf97b80>, <deap.creator.Individual object at 0x775ceac65c90>, <deap.creator.Individual object at 0x775ceac67ca0>, <deap.creator.Individual object at 0x775cef350cd0>, <deap.creator.Individual object at 0x775cef350cd0>, <deap.creator.Individual object at 0x775ceaba3460>, <deap.creator.Individual object at 0x775ceac67ca0>, <deap.creator.Individual object at 0x775ceafc9720>, <deap.creator.Individual object at 0x775ceae702b0>, <deap.creator.Individual object at 0x775cef350cd0>, <deap.creator.Individual object at 0x775ceac3c700>, <deap.creator.Individual object at 0x775cead57eb0>, <deap.creator.Individual object at 0x775ceacfd2a0>, <deap.creator.Individual object at 0x775ceaec89d0>, <deap.creator.Individual object at 0x775cef350cd0>, <deap.creator.Individual object at 0x775ceaba3460>, <deap